# Detección de Anomalías en Oleoductos de Transporte de Gasolina en la Costa Oriental del Estado Zulia Mediante Machine Learning

Este notebook presenta un proyecto completo desde cero para detectar anomalías en los oleoductos de transporte de gasolina utilizando un conjunto de datos sintético (`synthetic_oil_pipeline_monitoring.csv`).

**Estructura del Proyecto:**
- Selección del Dataset apropiado
  - Procesamiento, análisis e ingeniería de datos
- Selección del método apropiado
  - Entrenamiento, optimización (tuning) y predicción

## 1. Importación de Librerías
En esta etapa importamos todas las dependencias necesarias para el procesamiento, análisis y modelado de datos.

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import MinMaxScaler

In [48]:
# Cargar el DataFrame (asegúrate de ajustar la ruta del archivo si es necesario)
datos = pd.read_csv('synthetic_oil_pipeline_monitoring_updated.csv')

# Identificar las columnas numéricas
columnas_numericas = datos.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Eliminar filas con valores nulos
datos_limpios = datos.dropna()

# Crear una copia del DataFrame para normalización
datos_normalizados = datos_limpios.copy()

# Aplicar MinMaxScaler a las columnas numéricas
escalador = MinMaxScaler()
datos_normalizados[columnas_numericas] = escalador.fit_transform(datos_normalizados[columnas_numericas])

# Mostrar las primeras filas del DataFrame normalizado
print("Datos normalizados (sin valores nulos):")
print(datos_normalizados.head())

Datos normalizados (sin valores nulos):
             timestamp  pipeline_pressure_psi  flow_rate_bblh  temperature_c  \
0  2026-01-01 00:00:00               0.463149        0.462999       0.308603   
1  2026-01-01 00:15:00               0.444173        0.501835       0.476912   
3  2026-01-01 00:45:00               0.467673        0.493624       0.662930   
4  2026-01-01 01:00:00               0.442864        0.479890       0.211123   
5  2026-01-01 01:15:00               0.438636        0.482449       0.603332   

   is_anomaly  
0         0.0  
1         0.0  
3         0.0  
4         0.0  
5         0.0  


## 2. Selección del Dataset
Cargamos el conjunto de datos `synthetic_oil_pipeline_monitoring.csv`.

In [49]:
# Cargar el dataset actualizado con datos más realistas
df = pd.read_csv('synthetic_oil_pipeline_monitoring_updated.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Verificar valores nulos en el dataset
print("Valores nulos en el dataset:")
print(df.isnull().sum())

# Filtrar filas con valores nulos en 'is_anomaly'
df = df.dropna(subset=['is_anomaly']).copy()
print("\nDataset después de filtrar valores nulos en 'is_anomaly':")
print(df.head())

Valores nulos en el dataset:
timestamp                  0
pipeline_pressure_psi    100
flow_rate_bblh            60
temperature_c              0
is_anomaly                 0
dtype: int64

Dataset después de filtrar valores nulos en 'is_anomaly':
            timestamp  pipeline_pressure_psi  flow_rate_bblh  temperature_c  \
0 2026-01-01 00:00:00             712.653961     2392.663457      22.139549   
1 2026-01-01 00:15:00             695.120601     2597.011345      24.787552   
2 2026-01-01 00:30:00                    NaN     1911.896876      25.682480   
3 2026-01-01 00:45:00             716.833760     2553.807656      27.714188   
4 2026-01-01 01:00:00             693.911038     2481.540409      20.605887   

   is_anomaly  
0           0  
1           0  
2           0  
3           0  
4           0  


## 3. Procesamiento y Análisis de Datos (EDA)
Revisamos la información general, la presencia de valores nulos y la distribución de nuestra variable objetivo (`is_anomaly`).

In [50]:
# Manejo de valores nulos
print("Valores nulos antes de la imputación:")
print(df.isnull().sum())

# Imputar valores nulos en columnas numéricas con la media
df['pipeline_pressure_psi'].fillna(df['pipeline_pressure_psi'].mean(), inplace=True)
df['flow_rate_bblh'].fillna(df['flow_rate_bblh'].mean(), inplace=True)

# Eliminar filas con valores nulos en la columna 'is_anomaly'
df = df.dropna(subset=['is_anomaly']).copy()

# Verificar nuevamente los valores nulos
print("Valores nulos después de la imputación y filtrado:")
print(df.isnull().sum())

# Normalización de las columnas numéricas
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
numeric_columns = ['pipeline_pressure_psi', 'flow_rate_bblh', 'temperature_c']
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
# Verificar los datos normalizados
print("\nDatos normalizados:\n", df.head())

Valores nulos antes de la imputación:
timestamp                  0
pipeline_pressure_psi    100
flow_rate_bblh            60
temperature_c              0
is_anomaly                 0
dtype: int64
Valores nulos después de la imputación y filtrado:
timestamp                  0
pipeline_pressure_psi    100
flow_rate_bblh            60
temperature_c              0
is_anomaly                 0
dtype: int64

Datos normalizados:
             timestamp  pipeline_pressure_psi  flow_rate_bblh  temperature_c  \
0 2026-01-01 00:00:00               0.463149        0.463704       0.308603   
1 2026-01-01 00:15:00               0.444173        0.502489       0.476912   
2 2026-01-01 00:30:00                    NaN        0.372456       0.533794   
3 2026-01-01 00:45:00               0.467673        0.494289       0.662930   
4 2026-01-01 01:00:00               0.442864        0.480573       0.211123   

   is_anomaly  
0           0  
1           0  
2           0  
3           0  
4           0  


C:\Users\yoner\AppData\Local\Temp\ipykernel_15264\2871908452.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['pipeline_pressure_psi'].fillna(df['pipeline_pressure_psi'].mean(), inplace=True)
C:\Users\yoner\AppData\Local\Temp\ipykernel_15264\2871908452.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame 

In [ ]:
# Crear una nueva variable con los datos filtrados sin valores nulos en 'is_anomaly'
df_anomaly_filtered = df_normalized.copy()

# Convertir 'is_anomaly' a tipo entero para evitar problemas de visualización
df_anomaly_filtered['is_anomaly'] = df_anomaly_filtered['is_anomaly'].astype(int)

# Contar el número de casos de cada categoría en 'is_anomaly'
counts = df_anomaly_filtered['is_anomaly'].value_counts()

plt.figure(figsize=(6,4))
sns.countplot(data=df_anomaly_filtered, x='is_anomaly')
for i, count in enumerate(counts):
    plt.text(i, count + 0.02, f"{count} (filtrado)", ha='center', fontsize=12, color='black')
plt.title('Distribución de Anomalías (0: Operación Normal, 1: Anomalía)')
plt.ylim(0, counts.max() + 10)
plt.show()

KeyError: 'es_anomalia'

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(datos['marca_tiempo'][:500], datos['presion_oleoducto_psi'][:500], label='Presión (psi)', color='blue')
plt.plot(datos['marca_tiempo'][:500], datos['tasa_flujo_bblh'][:500], label='Tasa de Flujo (bbl/h)', color='green')
plt.title('Serie de Tiempo de Presión y Tasa de Flujo (Primeras 500 muestras)')
plt.xlabel('Tiempo')
plt.ylabel('Valor')
plt.legend()
plt.show()

## 4. Ingeniería de Características (Feature Engineering)
Para que el modelo pueda comprender mejor el contexto temporal, vamos a extraer características de la línea de tiempo, como la hora del día y el día de la semana. Luego aplicamos escalado de variables numéricas.

In [ ]:
datos['hora'] = datos['marca_tiempo'].dt.hour
datos['dia_semana'] = datos['marca_tiempo'].dt.dayofweek

# Variables predictoras (X) y objetivo (y)
X = datos.drop(['marca_tiempo', 'es_anomalia'], axis=1)
y = datos['es_anomalia']

X.head()

Dividimos los datos en conjuntos de entrenamiento (train) y prueba (test), y estandarizamos para un rendimiento óptimo de los métodos de Machine Learning.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Selección del Método
El problema corresponde a una clasificación binaria. Elegimos **Random Forest Classifier**, el cual es un método robusto apropiado para patrones no lineales en problemas de detección de anomalías espaciales y temporales basados en métricas físicas.

In [ ]:
rf = RandomForestClassifier(random_state=42)

## 6. Train and Tune (Entrenamiento y Optimización)
Aplicaremos `GridSearchCV` para buscar hiperparámetros que optimicen la clasificación (tuning).

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print('Mejores hiperparámetros encontrados:', grid_search.best_params_)

Asignamos el mejor estimador después de la optimización.

In [ ]:
best_model = grid_search.best_estimator_

## 7. Predicción y Evaluación
Realizamos las predicciones sobre el conjunto de pruebas y visualizamos las métricas de rendimiento.

In [ ]:
y_pred = best_model.predict(X_test_scaled)

print('Matriz de Confusión:')
print(confusion_matrix(y_test, y_pred))

print('\nReporte de Clasificación:')
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Seguro (No Anomalía)', 'Peligro (Anomalía)'], 
            yticklabels=['Seguro (No Anomalía)', 'Peligro (Anomalía)'])
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de Confusión')
plt.show()